### Bällebad – Lösung Schritt für Schritt

*Jugendwettbewerb Informatik 2025, 3. Runde, Junioraufgabe 1 (= 44. Bundeswettbewerb Informatik, 1. Runde, Junioraufgabe 1)*

In der Alan-Turing-Schule sollen für den Sportunterricht Materialien angeschafft werden. Während der Sportstunden
benötigt jede Schülerin und jeder Schüler einen eigenen Medizinball. Die Schulleitung möchte nun wissen, wie viele solcher
Bälle bestellt werden sollen. Aus dem Stundenplan ist bekannt, welche Klasse wann Sportunterricht hat und wie viele
Jugendliche in einer Klasse sind.

Beispiel:

| Klasse | Tag | Zeit | Klassenstärke |
|---|---|---|---|
| 6a | Montags | 14 – 16 | 32 |
| 6b | Dienstags | 14 – 16 | 27 |
| 7a | Montags | 13 – 15 | 28 |
| 7a | Dienstags | 13 – 15 | 28 |
| 8a | Montags | 15 – 17 | 26 |
| 8b | Freitags | 13 – 19 | 15 |

In diesem Beispiel werden maximal 60 Medizinbälle gleichzeitig benötigt, nämlich am Montag um 14 Uhr.

#### Junioraufgabe 1

Schreibe ein Programm, das Termine und Klassenstärken für den Sportunterricht einliest und die maximale Anzahl an
Medizinbällen ermittelt, die gleichzeitig benötigt werden. Außerdem soll es einen Zeitpunkt angeben, an welchem diese
Maximalzahl benötigt wird.

Wende dein Programm mindestens auf alle Beispiele an, die du auf den BWINF-Webseiten findest,
und dokumentiere die Ergebnisse.

#### Beispieldaten

Im Ordner `beispieldaten` stehen `ball00.txt` bis `ball07.txt` (`ball00.txt` ist das Beispiel aus der Aufgabenstellung).
Jede Datei ist so aufgebaut:

- 1. Zeile: Anzahl `n` der Unterrichtsstunden
- folgende `n` Zeilen: Klassenbezeichner, Wochentag, Startzeit, Endzeit, Klassengröße (durch Leerzeichen getrennt)

Zum Beispiel bedeutet `6a Dienstag 10 12 20`: Die Klasse 6a mit 20 Kindern hat am Dienstag von 10 bis 12 Uhr Unterricht.
Im Ordner `eigene_testdaten` liegen einige zusätzliche Sonderfälle.

## Was dir bei dieser Aufgabe hilft

**Algorithmische Kenntnisse**
- **Zählen mit einer Tabelle (Nutzungsplan):** Für jede Stunde der Woche gibt es einen Zähler, der am Anfang 0 ist. Jede
  Unterrichtseinheit erhöht die Zähler der Stunden, in denen sie stattfindet. Danach steht in der Tabelle für jede Stunde,
  wie viele Bälle gebraucht werden. Das ist eine einfache Form von **Bucketing** („Eimer“ für jede Stunde).
- **Maximum suchen:** Einmal durch die Tabelle gehen, den größten Wert merken und dann alle Stellen mit diesem Wert sammeln
  (lineare Suche).
- **Halboffene Zeiträume:** Eine Einheit von 13 bis 15 Uhr braucht die Bälle in den Stunden 13 und 14, aber nicht um 15 Uhr.
  Die Startzeit zählt mit, die Endzeit nicht. In Python macht `range(start, ende)` genau das.
- **Zwei Dimensionen:** Die Tabelle hat eine Zeile pro Wochentag und eine Spalte pro Stunde (Liste von Listen).

**Nützliche Funktionen** (Bibliotheken braucht man keine)
- `liste.index(wert)` liefert die Position eines Wertes in einer Liste (hier: Nummer des Wochentags).
- `range(start, ende)`, `max(...)`, `split()`.
- Für Fortgeschrittene: Statt einer Tabelle mit 24 Stunden pro Tag kann man auch nur die Zeitpunkte betrachten, an denen
  eine Einheit beginnt oder endet (Ereignisse sortieren, Zähler hoch und runter). Das braucht `sorted` und ist bei
  Zeiten in Minuten viel kürzer als eine Tabelle mit allen Minuten (*Sweep-Line*-Verfahren).

## Die Idee

Wir bauen einen **Nutzungsplan**: eine Tabelle mit einer Zeile für jeden Wochentag und einer Spalte für jede Stunde (0 bis 23).
Jede Zelle sagt, wie viele Bälle in dieser Stunde gebraucht werden. Am Anfang steht überall 0.

Für jede Unterrichtseinheit zählen wir die Klassengröße in allen Stunden dazu, in denen die Einheit stattfindet.
Danach suchen wir das Maximum der Tabelle und alle Stunden, in denen es vorkommt.

Das Beispiel `ball00.txt` von Hand für Montag: 6a (32 Kinder) hat von 14 bis 16 Uhr Unterricht, 7a (28) von 13 bis 15 Uhr,
8a (26) von 15 bis 17 Uhr.

| Stunde | 13 | 14 | 15 | 16 |
|---|---|---|---|---|
| Bälle | 28 | 32 + 28 = **60** | 32 + 26 = 58 | 26 |

Beachte die Endzeit: Die Klasse 7a hört um 15 Uhr auf, die Klasse 8a fängt um 15 Uhr an. Die Bälle können einfach
weitergegeben werden, deshalb zählt 7a um 15 Uhr nicht mehr mit.

## Schritt 1: Die Datei einlesen

Die erste Zeile enthält die Anzahl `n` der Unterrichtseinheiten, danach folgt je eine Zeile pro Einheit.
Mit `split()` zerlegen wir eine Zeile an den Leerzeichen. Aus `'6a Montag 14 16 32'` wird die Liste
`['6a', 'Montag', '14', '16', '32']`. Start, Ende und Klassengröße wandeln wir mit `int(...)` in Zahlen um.
Jede Einheit speichern wir als eine kleine Liste `[klasse, tag, start, ende, groesse]`.

In [ ]:
def lies_stundenplan(dateiname):
    einheiten = []
    with open(dateiname, encoding='utf-8') as f:
        n = int(f.readline())
        for i in range(n):
            teile = f.readline().split()
            klasse = teile[0]
            tag = teile[1]
            start = int(teile[2])
            ende = int(teile[3])
            groesse = int(teile[4])
            einheiten.append([klasse, tag, start, ende, groesse])
    return einheiten


einheiten = lies_stundenplan('beispieldaten/ball00.txt')
for einheit in einheiten:
    print(einheit)

['6a', 'Montag', 14, 16, 32]
['6b', 'Dienstag', 14, 16, 27]
['7a', 'Montag', 13, 15, 28]
['7a', 'Dienstag', 13, 15, 28]
['8a', 'Montag', 15, 17, 26]
['8b', 'Freitag', 13, 19, 15]


## Schritt 2: Der Nutzungsplan

Die Wochentage stehen in einer Liste. Der Tag `'Montag'` hat die Nummer 0 (`WOCHENTAGE.index('Montag')`), der Tag
`'Dienstag'` die Nummer 1 und so weiter. Der Plan hat für jeden Tag eine Liste mit 24 Nullen, `plan[t][stunde]` ist die Anzahl
der Bälle am Tag `t` in der Stunde, die mit `stunde` Uhr beginnt.

Die Zeile `plan.append([0] * 24)` steht in einer Schleife, so bekommt jeder Tag seine **eigene** Liste.
(Mit `[[0] * 24] * 7` hätten alle Tage dieselbe Liste, und eine Änderung am Montag würde auch alle anderen Tage ändern.)

Danach tragen wir jede Einheit ein. `range(start, ende)` enthält die Startstunde, aber nicht die Endstunde,
genau wie gewünscht.

In [ ]:
WOCHENTAGE = ['Montag', 'Dienstag', 'Mittwoch', 'Donnerstag', 'Freitag', 'Samstag', 'Sonntag']


def erstelle_plan(einheiten):
    plan = []
    for tag in WOCHENTAGE:
        plan.append([0] * 24)               # 24 Stunden, am Anfang werden keine Bälle gebraucht

    for klasse, tag, start, ende, groesse in einheiten:
        t = WOCHENTAGE.index(tag)           # Nummer des Wochentags
        for stunde in range(start, ende):   # Startstunde zählt, Endstunde nicht
            plan[t][stunde] = plan[t][stunde] + groesse
    return plan


plan = erstelle_plan(einheiten)
print('Montag, Stunden 12 bis 17:', plan[0][12:18])
print('Dienstag, Stunden 12 bis 17:', plan[1][12:18])
print('Freitag, Stunden 12 bis 19:', plan[4][12:20])

Montag, Stunden 12 bis 17: [0, 28, 60, 58, 26, 0]
Dienstag, Stunden 12 bis 17: [0, 28, 55, 27, 0, 0]
Freitag, Stunden 12 bis 19: [0, 15, 15, 15, 15, 15, 15, 0]


Die Montagszeile stimmt mit unserer Rechnung von oben überein (0, 28, 60, 58, 26, 0).
Der Plan dient auch gleich als Dokumentation: Man sieht für jede Stunde, wie viele Bälle gebraucht werden.

## Schritt 3: Das Maximum finden

Wir gehen zweimal durch den ganzen Plan:

1. Beim ersten Durchgang suchen wir den größten Wert (`maximum`).
2. Beim zweiten Durchgang sammeln wir alle Zeitpunkte, an denen genau dieser Wert steht.
   Die Aufgabe verlangt nur einen Zeitpunkt, aber für die Schulleitung ist es nützlich, alle zu kennen.

Ist nie ein Ball nötig (leerer Stundenplan), ist das Maximum 0. Dann geben wir keine Zeitpunkte an,
denn „0 Bälle werden gebraucht“ gilt zu jeder Zeit.

In [ ]:
def finde_maximum(plan):
    maximum = 0
    for t in range(len(plan)):
        for stunde in range(24):
            if plan[t][stunde] > maximum:
                maximum = plan[t][stunde]

    zeitpunkte = []
    if maximum > 0:
        for t in range(len(plan)):
            for stunde in range(24):
                if plan[t][stunde] == maximum:
                    zeitpunkte.append([t, stunde])
    return maximum, zeitpunkte


maximum, zeitpunkte = finde_maximum(plan)
print(maximum, zeitpunkte)

60 [[0, 14]]


## Schritt 4: Ausgabe

Aus `[0, 14]` machen wir den Text „Montag um 14 Uhr“. Das `:02` im f-String füllt die Stunde mit einer führenden Null
auf zwei Stellen auf („um 09 Uhr“), so wie in den Lösungshinweisen.

In [ ]:
def bearbeite(dateiname):
    einheiten = lies_stundenplan(dateiname)
    plan = erstelle_plan(einheiten)
    maximum, zeitpunkte = finde_maximum(plan)
    print(f'{dateiname}: Anzahl der Einheiten: {len(einheiten)}')
    print(f'  Maximal benötigte Medizinbälle: {maximum}')
    if maximum == 0:
        print('  (es werden keine Bälle benötigt)')
    else:
        orte = []
        for t, stunde in zeitpunkte:
            orte.append(f'{WOCHENTAGE[t]} um {stunde:02} Uhr')
        print('  Zu diesen Zeiten:', ', '.join(orte))


bearbeite('beispieldaten/ball00.txt')

beispieldaten/ball00.txt: Anzahl der Einheiten: 6
  Maximal benötigte Medizinbälle: 60
  Zu diesen Zeiten: Montag um 14 Uhr


## Schritt 5: Alle Beispiele

`', '.join(orte)` hängt die Texte der Liste mit „, “ dazwischen zusammen.

In [ ]:
for nr in range(8):
    bearbeite(f'beispieldaten/ball0{nr}.txt')

beispieldaten/ball00.txt: Anzahl der Einheiten: 6
  Maximal benötigte Medizinbälle: 60
  Zu diesen Zeiten: Montag um 14 Uhr
beispieldaten/ball01.txt: Anzahl der Einheiten: 31
  Maximal benötigte Medizinbälle: 96
  Zu diesen Zeiten: Mittwoch um 11 Uhr
beispieldaten/ball02.txt: Anzahl der Einheiten: 62
  Maximal benötigte Medizinbälle: 157
  Zu diesen Zeiten: Montag um 10 Uhr
beispieldaten/ball03.txt: Anzahl der Einheiten: 65
  Maximal benötigte Medizinbälle: 152
  Zu diesen Zeiten: Freitag um 09 Uhr
beispieldaten/ball04.txt: Anzahl der Einheiten: 17
  Maximal benötigte Medizinbälle: 31
  Zu diesen Zeiten: Montag um 10 Uhr, Montag um 11 Uhr
beispieldaten/ball05.txt: Anzahl der Einheiten: 12
  Maximal benötigte Medizinbälle: 60
  Zu diesen Zeiten: Donnerstag um 08 Uhr, Donnerstag um 09 Uhr
beispieldaten/ball06.txt: Anzahl der Einheiten: 8
  Maximal benötigte Medizinbälle: 90
  Zu diesen Zeiten: Dienstag um 09 Uhr, Dienstag um 10 Uhr
beispieldaten/ball07.txt: Anzahl der Einheiten: 18
  Max

Die Ergebnisse stimmen mit den offiziellen Lösungshinweisen überein, auch dort, wo es mehrere Zeitpunkte mit
dem Maximum gibt (`ball04` bis `ball07`). Für die Aufgabe genügt ein Zeitpunkt je Beispiel, hier sind alle angegeben.

### Sonderfälle
Im Ordner `eigene_testdaten` liegen einige Fälle, bei denen man leicht Fehler macht:

- `test01`: Zwei Klassen mit je 30 Kindern haben gleichzeitig Unterricht, später drei mit je 12. Es müssen die Klassengrößen
  **summiert** werden (60 Bälle), nicht die Anzahl der Klassen.
- `test02`: Eine lange Einheit (13 bis 19 Uhr) wird von mehreren kurzen überlagert. Die Bälle der langen Einheit müssen in
  **jeder** Stunde mitgezählt werden, nicht nur zu Beginn.
- `test03`: Es gibt nur eine einzige Einheit.
- `test04`: Der Stundenplan ist leer (`0`).
- `test05`: Eine Einheit hat die Dauer 0 (Start gleich Ende) und braucht keine Bälle.

In [ ]:
for name in ['test01_summe_statt_anzahl', 'test02_umschliessende_stunde', 'test03_eine_zeile',
             'test04_leer', 'test05_ohne_dauer']:
    bearbeite(f'eigene_testdaten/{name}.txt')

eigene_testdaten/test01_summe_statt_anzahl.txt: Anzahl der Einheiten: 5
  Maximal benötigte Medizinbälle: 60
  Zu diesen Zeiten: Montag um 08 Uhr, Montag um 09 Uhr
eigene_testdaten/test02_umschliessende_stunde.txt: Anzahl der Einheiten: 4
  Maximal benötigte Medizinbälle: 55
  Zu diesen Zeiten: Freitag um 15 Uhr
eigene_testdaten/test03_eine_zeile.txt: Anzahl der Einheiten: 1
  Maximal benötigte Medizinbälle: 24
  Zu diesen Zeiten: Mittwoch um 09 Uhr
eigene_testdaten/test04_leer.txt: Anzahl der Einheiten: 0
  Maximal benötigte Medizinbälle: 0
  (es werden keine Bälle benötigt)
eigene_testdaten/test05_ohne_dauer.txt: Anzahl der Einheiten: 2
  Maximal benötigte Medizinbälle: 12
  Zu diesen Zeiten: Montag um 09 Uhr, Montag um 10 Uhr


## Für Neugierige: Alles in einer Liste

Man kann den Nutzungsplan auch **ohne Wochentage** als eine einzige lange Liste mit 7 · 24 Einträgen bauen:
Der Eintrag für Tag `t` und Stunde `h` steht an der Position `t * 24 + h`. Das Programm wird dadurch etwas kürzer,
aber die Ausgabe der Zeitpunkte etwas umständlicher (`//` und `%` rechnen die Position wieder in Tag und Stunde um).
Zum Vergleich: Beide Verfahren müssen dasselbe Ergebnis liefern.

In [ ]:
def maximum_flach(dateiname):
    woche = [0] * (7 * 24)
    for klasse, tag, start, ende, groesse in lies_stundenplan(dateiname):
        t = WOCHENTAGE.index(tag)
        for stunde in range(start, ende):
            woche[t * 24 + stunde] = woche[t * 24 + stunde] + groesse
    return max(woche), woche.index(max(woche))       # Maximum und erste Position, an der es steht


for nr in range(8):
    dateiname = f'beispieldaten/ball0{nr}.txt'
    maximum, position = maximum_flach(dateiname)
    tag, stunde = position // 24, position % 24
    print(f'ball0{nr}: {maximum} Bälle, erstmals {WOCHENTAGE[tag]} um {stunde:02} Uhr')

ball00: 60 Bälle, erstmals Montag um 14 Uhr
ball01: 96 Bälle, erstmals Mittwoch um 11 Uhr
ball02: 157 Bälle, erstmals Montag um 10 Uhr
ball03: 152 Bälle, erstmals Freitag um 09 Uhr
ball04: 31 Bälle, erstmals Montag um 10 Uhr
ball05: 60 Bälle, erstmals Donnerstag um 08 Uhr
ball06: 90 Bälle, erstmals Dienstag um 09 Uhr
ball07: 59 Bälle, erstmals Freitag um 10 Uhr
